# Genome PRD v1 — 01 Training dataset

Validate the materialized v2 dataset against the canonical 25-feature PRD contract. If the artifact is absent, first run `python -m src.features.materialize` from the repository root.

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd
import pyarrow.parquet as pq

here = Path.cwd().resolve()
ROOT = next(path for path in (here, *here.parents) if (path / 'src/training/prd_config.py').is_file())
sys.path.insert(0, str(ROOT))
from src.features.genome import GENOME_FEATURE_COLUMNS
from src.training import prd_config
from src.training.modeling import validate_v2

## Contract and artifact schema

In [2]:
assert len(prd_config.PRD_FEATURES) == 25
assert prd_config.FEATURE_ARTIFACT_PATH.is_file(), 'Run: python -m src.features.materialize'
metadata = json.loads(prd_config.FEATURE_METADATA_PATH.read_text())
schema = pq.ParquetFile(prd_config.FEATURE_ARTIFACT_PATH).schema_arrow
assert metadata['predictorNames'] == list(prd_config.PRD_FEATURES)
assert metadata['rowCount'] == pq.ParquetFile(prd_config.FEATURE_ARTIFACT_PATH).metadata.num_rows
assert all(name in schema.names for name in prd_config.PRD_FEATURES)
assert {name: metadata['outputSchema'][name] for name in prd_config.PRD_FEATURES} == prd_config.PRD_FEATURE_DTYPES
display(pd.DataFrame({'feature': prd_config.PRD_FEATURES, 'dtype': [prd_config.PRD_FEATURE_DTYPES[name] for name in prd_config.PRD_FEATURES]}))

,feature,dtype
0,global_rating_count,uint64
1,global_mean_rating,float32
2,user_rating_count,uint64
3,user_mean_rating,float32
4,user_rating_std_pop,float32
5,user_seconds_since_last_rating,float64
6,movie_rating_count,uint64
7,movie_mean_rating,float32
8,movie_rating_std_pop,float32
9,movie_rating_count_30d,uint64


## Full artifact validation and summary

In [3]:
validation = validate_v2(None, prd_config.FEATURE_ARTIFACT_PATH, prd_config.FEATURE_METADATA_PATH, prd_config.RATINGS_SOURCE_PATH)
display(pd.Series({
    'rows': validation['row_count'],
    'predictors': validation['predictor_count'],
    'target_prevalence': validation['target_prevalence'],
    'rating_event_ids_complete_unique': validation['rating_event_ids_complete_unique'],
    'timestamps_nondecreasing': validation['timestamps_nondecreasing'],
}).to_frame('value'))
display(pd.Series(validation['genome_null_rates'], name='null_rate').rename_axis('Genome feature').to_frame())
assert set(validation['genome_null_rates']) == set(GENOME_FEATURE_COLUMNS)

,value
rows,20000263
predictors,25
target_prevalence,0.499764
rating_event_ids_complete_unique,True
timestamps_nondecreasing,True


,null_rate
Genome feature,
genome_user_positive_cosine,0.034133
genome_user_negative_cosine,0.032089
genome_preference_margin,0.042514
genome_nearest_liked_similarity,0.034133
genome_top5_liked_similarity,0.034133
genome_movie_relevance_mean,0.009991
genome_movie_relevance_std,0.009991
genome_movie_top10_mean,0.009991
